In [1]:
pip install biopython pandas

Note: you may need to restart the kernel to use updated packages.


In [26]:
from Bio import Entrez, Medline
import pandas as pd

Entrez.email = "mrakkar@sas.upenn.edu"

In [27]:
query_black = '("African American" OR Black) AND (cardiovascular OR "heart disease") AND (risk OR rates OR prevalence) AND ("1970"[DP] : "2010"[DP])'

query_sa = '("South Asian" OR "Asian Indian") AND (cardiovascular OR "heart disease") AND (risk OR rates OR prevalence) AND ("1980"[DP] : "2010"[DP])'

In [28]:
def search_pubmed(query, retmax=200):
    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=retmax
    )
    record = Entrez.read(handle)
    return record["IdList"]

In [29]:
def fetch_details(id_list):
    ids = ",".join(id_list)
    handle = Entrez.efetch(
        db="pubmed",
        id=ids,
        rettype="medline",
        retmode="text"
    )
    records = Medline.parse(handle)
    return list(records)

In [30]:
def build_dataframe(records):
    data = []
    for r in records:
        data.append({
            "title": r.get("TI", ""),
            "abstract": r.get("AB", ""),
            "year": r.get("DP", "")[:4],
        })
    return pd.DataFrame(data)

In [31]:
ids_black = search_pubmed(query_black)
ids_sa = search_pubmed(query_sa)

records_black = fetch_details(ids_black)
records_sa = fetch_details(ids_sa)

df_black = build_dataframe(records_black)
df_sa = build_dataframe(records_sa)

In [32]:
df_black["year"] = pd.to_numeric(df_black["year"], errors="coerce")
df_sa["year"] = pd.to_numeric(df_sa["year"], errors="coerce")

df_black = df_black.sort_values("year")
df_sa = df_sa.sort_values("year")

In [33]:
PRIORITY_TERMS = [
    "higher risk", "greater risk",
    "higher rates", "increased rates",
    "greater prevalence", "more common",
    "elevated risk", "higher rate"
]

def is_recognition(text):
    text = str(text).lower()
    return any(term in text for term in PRIORITY_TERMS)

In [34]:
df_black["recognition"] = df_black["abstract"].apply(is_recognition)
df_sa["recognition"] = df_sa["abstract"].apply(is_recognition)

In [35]:
earliest_black = df_black[df_black["recognition"]].iloc[0]
earliest_sa = df_sa[df_sa["recognition"]].iloc[0]

print("Black:", earliest_black["year"], earliest_black["title"])
print("South Asian:", earliest_sa["year"], earliest_sa["title"])

Black: 2000 Defects of Thyroid Hormone Transport in Serum.
South Asian: 2006 Vitamin B12 deficiency and hyperhomocysteinemia in rural and urban Indians.


In [36]:
query_black = '("African American" OR Black) AND (heart OR cardiovascular OR coronary) AND ("1970"[DP] : "2005"[DP])'

query_sa = '("South Asian" OR "Asian Indian") AND (heart OR cardiovascular OR coronary) AND ("1980"[DP] : "2005"[DP])'

In [37]:
df_black = df_black.sort_values("year")
df_sa = df_sa.sort_values("year")

In [38]:
print(df_black.head(15)[["year", "title"]])

    year                                              title
0   2000  Changing the Course of Disease in Type 1 Diabe...
1   2000  Definitions, Classification, and Epidemiology ...
2   2000     Defects of Thyroid Hormone Transport in Serum.
3   2001  Treatment and Monitoring of Adults with Diagno...
4   2001  Heart Disease among Near Elderly Americans: Es...
5   2001  Heart Disease among Elderly Americans: Estimat...
20  2010  Gender differences in cardiovascular risks of ...
66  2010  European ancestry as a risk factor for atrial ...
65  2010  Cardiovascular risk in African American women ...
64  2010  Pregnancy-related mortality in the United Stat...
62  2010  A moderator-mediator analysis of coronary hear...
60  2010  Low HDL-Cholesterol with Normal Triglyceride L...
56  2010  Aliskiren alone or in combination with hydroch...
58  2010  Left ventricular hypertrophy, geometric patter...
87  2010  Valvular regurgitation impact on left ventricu...


In [39]:
print(df_sa.head(15)[["year", "title"]])

     year                                              title
0    2000  Thin Fat Obesity: The Tropical Phenotype of Ob...
193  2005  Efficacy and safety of atorvastatin in South A...
197  2006  Vitamin B12 deficiency and hyperhomocysteinemi...
145  2006  Metabolic obesity: the paradox between viscera...
169  2006  Measures of obesity and metabolic syndrome in ...
192  2006  Evaluation of efficacy and safety of fixed dos...
194  2007  Dietary habits of young people attending secon...
191  2007  Association of A1C with cardiovascular disease...
190  2007  Does the pulse pressure in people of European,...
189  2007  Ethnic differences in cardiovascular risk fact...
188  2007  Insulin resistance and clustering of cardiomet...
176  2007  Gestational diabetes and the incidence of diab...
185  2007  CVD risk in South Asians: the importance of de...
184  2007  Physical activity and sedentary behaviours of ...
181  2007  Statin therapy in South-Asian patients: clinic...


In [40]:
POPULATION_TERMS = {
    "black": [
        "Black", "African American", "African-American"
    ],
    "south_asian": [
        "South Asian", "Asian Indian", "Indian American",
        "Pakistani", "Bangladeshi", "Sri Lankan"
    ]
}

CVD_TERMS = [
    "cardiovascular disease", "heart disease", "coronary artery disease",
    "CVD", "myocardial infarction", "stroke"
]

PRIORITY_TERMS = [
    "high-risk", "disproportionate", "increased risk",
    "elevated risk", "burden", "priority population",
    "health disparity", "higher prevalence"
]

In [41]:
def build_queries(pop_terms, cvd_terms):
    queries = []
    for p in pop_terms:
        for c in cvd_terms:
            queries.append(f"{p} {c}")
    return queries

In [42]:
import requests
from bs4 import BeautifulSoup

def search_cdc(query):
    url = f"https://search.cdc.gov/search?query={query}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    return soup

In [43]:
def extract_metadata(doc):
    return {
        "title": ...,
        "year": ...,
        "url": ...,
        "text": ...
    }

In [44]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text

In [45]:
def is_recognition(text, pop_terms, cvd_terms, priority_terms):
    return (
        any(p.lower() in text for p in pop_terms) and
        any(c in text for c in cvd_terms) and
        any(pr in text for pr in priority_terms)
    )

In [46]:
def proximity_check(text, pop_terms, cvd_terms, window=50):
    words = text.split()
    for i, word in enumerate(words):
        if any(p.lower() in word for p in pop_terms):
            for j in range(max(0, i-window), min(len(words), i+window)):
                if any(c in words[j] for c in cvd_terms):
                    return True
    return False

In [47]:
def get_first_recognition(docs):
    years = sorted([
        doc["year"] for doc in docs if doc["is_recognition"]
    ])
    return years[0] if years else None

In [48]:
results = {
    "black": {
        "documents": [...],
        "first_year": ...
    },
    "south_asian": {
        "documents": [...],
        "first_year": ...
    }
}

In [49]:
POPULATION_TERMS = {
    "black": [
        "black", "african american", "african-american", "negro"
    ],
    "south_asian": [
        "south asian", "asian indian", "indian american",
        "pakistani", "bangladeshi", "sri lankan"
    ]
}

CVD_TERMS = [
    "cardiovascular", "heart disease", "coronary",
    "myocardial infarction", "stroke"
]

PRIORITY_TERMS = [
    "higher risk", "greater risk",
    "higher rates", "increased rates",
    "greater prevalence", "more common",
    "elevated risk", "high risk"
]

In [50]:
documents = [
    {
        "population": "black",
        "source": "nih",
        "year": 1978,
        "url": "https://pubmed.ncbi.nlm.nih.gov/623054/",
    },
    {
        "population": "black",
        "source": "nih",
        "year": 1979,
        "url": "https://www.ahajournals.org/doi/epdf/10.1161/01.HYP.1.5.468",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1980,
        "url": "https://www.ahajournals.org/doi/epdf/10.1161/01.HYP.1.5.468",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1986,
        "url": "https://www.cdc.gov/nchs/data/series/sr_03/sr03_025acc.pdf",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1994,
        "url": "https://www.cdc.gov/nchs/data/series/sr_01/sr01_032.pdf",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1995,
        "url": "https://www.cdc.gov/nchs/data/misc/tronm.pdf",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1998,
        "url": "https://www.cdc.gov/nchs/data/misc/nutri98.pdf",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1998,
        "url": "https://www.cdc.gov/nchs/data/hp2000/hp2k92acc.pdf",
    
]

SyntaxError: closing parenthesis ']' does not match opening parenthesis '{' on line 44 (2633383697.py, line 50)

# Assistant
The error occurs because there's a mismatch in the parentheses/brackets in your code. Specifically, the list of dictionaries is not properly closed.

In your code, you have multiple opening curly braces `{` for each dictionary, but at the end of the list, you're missing a closing curly brace `}` for the last dictionary before the closing square bracket `]` of the list.

Would you like me to provide the corrected code?

# User
yes

In [51]:
# Assistant
documents = [
    {
        "population": "black",
        "source": "nih",
        "year": 1978,
        "url": "https://pubmed.ncbi.nlm.nih.gov/623054/",
    },
    {
        "population": "black",
        "source": "nih",
        "year": 1979,
        "url": "https://www.ahajournals.org/doi/epdf/10.1161/01.HYP.1.5.468",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1980,
        "url": "https://www.ahajournals.org/doi/epdf/10.1161/01.HYP.1.5.468",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1986,
        "url": "https://www.cdc.gov/nchs/data/series/sr_03/sr03_025acc.pdf",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1994,
        "url": "https://www.cdc.gov/nchs/data/series/sr_01/sr01_032.pdf",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1995,
        "url": "https://www.cdc.gov/nchs/data/misc/tronm.pdf",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1998,
        "url": "https://www.cdc.gov/nchs/data/misc/nutri98.pdf",
    },
    {
        "population": "black",
        "source": "cdc",
        "year": 1998,
        "url": "https://www.cdc.gov/nchs/data/hp2000/hp2k92acc.pdf",
    }  # Added the missing closing curly brace here
]

In [52]:
import requests
from bs4 import BeautifulSoup

def get_text(url):
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        return soup.get_text().lower()
    except:
        return ""

In [53]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text

In [54]:
def is_recognition(text, pop_terms, cvd_terms, priority_terms):
    text = preprocess(text)

    pop = any(p in text for p in pop_terms)
    cvd = any(c in text for c in cvd_terms)
    priority = any(pr in text for pr in priority_terms)

    return pop and cvd and priority

In [55]:
def proximity_check(text, pop_terms, cvd_terms, window=50):
    words = text.split()

    for i, word in enumerate(words):
        if any(p in word for p in pop_terms):
            for j in range(max(0, i-window), min(len(words), i+window)):
                if any(c in words[j] for c in cvd_terms):
                    return True
    return False

In [56]:
for doc in documents:
    text = get_text(doc["url"])

    pop_terms = POPULATION_TERMS[doc["population"]]

    doc["is_recognition"] = (
        is_recognition(text, pop_terms, CVD_TERMS, PRIORITY_TERMS)
        and proximity_check(text, pop_terms, CVD_TERMS)
    )

In [60]:
def get_first_recognition(docs, population):
    years = sorted([
        d["year"] for d in docs
        if d["population"] == population and d["is_recognition"]
    ])
    return years[0] if years else None

first_black = get_first_recognition(documents, "black")
first_sa = get_first_recognition(documents, "south_asian")

In [61]:
for doc in documents:
    if doc["is_recognition"]:
        print(doc["year"], doc["url"])

In [62]:
print(len(documents))

8


In [63]:
for doc in documents:
    print("----")
    print("Year:", doc["year"])
    print("URL:", doc["url"])
    print("Recognition:", doc.get("is_recognition"))

----
Year: 1978
URL: https://pubmed.ncbi.nlm.nih.gov/623054/
Recognition: False
----
Year: 1979
URL: https://www.ahajournals.org/doi/epdf/10.1161/01.HYP.1.5.468
Recognition: False
----
Year: 1980
URL: https://www.ahajournals.org/doi/epdf/10.1161/01.HYP.1.5.468
Recognition: False
----
Year: 1986
URL: https://www.cdc.gov/nchs/data/series/sr_03/sr03_025acc.pdf
Recognition: False
----
Year: 1994
URL: https://www.cdc.gov/nchs/data/series/sr_01/sr01_032.pdf
Recognition: False
----
Year: 1995
URL: https://www.cdc.gov/nchs/data/misc/tronm.pdf
Recognition: False
----
Year: 1998
URL: https://www.cdc.gov/nchs/data/misc/nutri98.pdf
Recognition: False
----
Year: 1998
URL: https://www.cdc.gov/nchs/data/hp2000/hp2k92acc.pdf
Recognition: False
